# V2 raw RTF to paragraph Parquet

This notebook runs the first V2 chain handler only. It converts one raw RTF document into normalized, globally numbered paragraphs and writes:

- `paragraphs.parquet` — one row per paragraph;
- `numbered_document.json` — the complete numbered text and paragraph count.

No language model or GPU is required. The later classification and extraction handlers are intentionally not run here.

## Dependencies

Use the project environment when running locally. In a clean Colab runtime, uncomment and run the installation command.

In [ ]:
# Colab only:
# %pip install -q "pyarrow>=24,<25" "striprtf>=0.0.32,<0.0.33" google-cloud-storage

## Import the V2 preprocessing handler

The bootstrap supports starting Jupyter from the repository root or any directory beneath it.

In [ ]:
from pathlib import Path
import sys

import pyarrow.parquet as pq
from IPython.display import display

cwd = Path.cwd().resolve()
repository_root = None
source_root = None

for candidate in (cwd, *cwd.parents):
    candidate_source = candidate / "src"
    if (candidate_source / "document_split" / "__init__.py").exists():
        repository_root = candidate
        source_root = candidate_source
        break
    if (candidate / "document_split" / "__init__.py").exists():
        source_root = candidate
        repository_root = candidate.parent
        break

if source_root is None or repository_root is None:
    raise RuntimeError(
        "Could not locate src/document_split. Run the notebook from the "
        "cloned repository or add its src directory to sys.path."
    )

if str(source_root) not in sys.path:
    sys.path.insert(0, str(source_root))

from document_split.v2 import (
    RtfToParagraphParquetHandler,
    V2DocumentState,
    V2_INFO_VERSION,
)

print(f"Repository: {repository_root}")
print(f"V2 version: {V2_INFO_VERSION}")

## Configure the source document

Use `SOURCE_MODE = "local"` for a file already on disk. Use `SOURCE_MODE = "gcs"` to download the standard `{justice_kind}/{document_id}.rtf` object from Cloud Storage.

In [ ]:
SOURCE_MODE = "local"  # "local" or "gcs"

JUSTICE_KIND = 2
DOCUMENT_ID = "117888886"

# Local source configuration
LOCAL_RTF_PATH = repository_root / f"{DOCUMENT_ID}.rtf"

# GCS source configuration; required only when SOURCE_MODE == "gcs"
GCS_SOURCE_BUCKET = ""
GCS_SOURCE_OBJECT = f"{JUSTICE_KIND}/{DOCUMENT_ID}.rtf"

# Local artifacts preserve the V2 cloud-style directory structure.
OUTPUT_DIR = (
    repository_root
    / "artifacts"
    / V2_INFO_VERSION
    / str(JUSTICE_KIND)
    / DOCUMENT_ID
)

print(f"Source mode: {SOURCE_MODE}")
print(f"Output directory: {OUTPUT_DIR}")

## Load the raw RTF

In [ ]:
if SOURCE_MODE == "local":
    if not LOCAL_RTF_PATH.is_file():
        raise FileNotFoundError(f"RTF file does not exist: {LOCAL_RTF_PATH}")
    raw_rtf = LOCAL_RTF_PATH.read_bytes()
elif SOURCE_MODE == "gcs":
    if not GCS_SOURCE_BUCKET:
        raise ValueError("Set GCS_SOURCE_BUCKET before using GCS mode")
    from google.cloud import storage

    storage_client = storage.Client()
    raw_rtf = (
        storage_client.bucket(GCS_SOURCE_BUCKET)
        .blob(GCS_SOURCE_OBJECT)
        .download_as_bytes()
    )
else:
    raise ValueError("SOURCE_MODE must be either 'local' or 'gcs'")

print(f"Loaded {len(raw_rtf):,} bytes for document {DOCUMENT_ID}")

## Run the RTF-to-Parquet handler

This handler performs the same RTF cleanup, paragraph splitting, and global numbering used by the complete V2 pipeline. It does not use the handler context, tokenizer, or model.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

state = V2DocumentState(
    document_id=DOCUMENT_ID,
    justice_kind=JUSTICE_KIND,
    raw_rtf=raw_rtf,
    work_dir=OUTPUT_DIR,
)

preparation_handler = RtfToParagraphParquetHandler()
state = preparation_handler.transform(state, context=None)

paragraphs_path = state.artifact_path("paragraphs.parquet")
numbered_document_path = state.artifact_path("numbered_document.json")

print(f"Paragraphs: {len(state.paragraphs):,}")
print(f"Parquet: {paragraphs_path}")
print(f"Numbered JSON: {numbered_document_path}")

## Validate and inspect the Parquet output

The file intentionally has one row per paragraph. The final extraction pipeline later merges handler results into one document-level row.

In [ ]:
paragraph_table = pq.read_table(paragraphs_path, use_threads=False)

expected_columns = [
    "document_id",
    "paragraph_index",
    "paragraph_order",
    "numbered_text",
    "text",
]
assert paragraph_table.column_names == expected_columns
assert paragraph_table.num_rows == len(state.paragraphs)
assert paragraph_table.column("paragraph_index").to_pylist() == list(
    range(1, paragraph_table.num_rows + 1)
)

print(paragraph_table.schema)
display(paragraph_table.to_pandas().head(20))

## Preview the numbered document

In [ ]:
preview_characters = 5_000
print(state.numbered_text[:preview_characters])
if len(state.numbered_text) > preview_characters:
    print("\n... preview truncated ...")